# Clinical Trial Semantic Search

Run the cells from top to bottom. Keep **ctg-studies(1).csv** in the same folder as this notebook, or edit `CSV_PATH`.

The notebook uses the actual CSV columns and derives year only from **Start Date**. No years are fabricated. Missing dates are included in unfiltered searches. These notebooks search trial records; they do not measure changes in treatment efficacy across years.

The first run downloads two pretrained embedding models. Restart the kernel after package installation if needed. Each CSV row is a searchable document. Long records may be truncated to each model's input limit during embedding; returned results show the full source text. Model results are presented separately.


In [ ]:
%pip install -U pandas sentence-transformers chromadb gradio

In [ ]:
from pathlib import Path
import pandas as pd

# Keep the updated CSV beside this notebook, or enter its full path here.
CSV_PATH = Path("../chromadb/ctg-studies.csv")
# The upload/ fallback also supports running from the parent folder.
if not CSV_PATH.is_file() and (Path("upload") / CSV_PATH).is_file():
    CSV_PATH = Path("upload") / CSV_PATH
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        "Place ctg-studies(1).csv beside this notebook or edit CSV_PATH."
    )

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
required = ["NCT Number", "Study Title", "Brief Summary", "Start Date"]
missing = [name for name in required if name not in df.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

text_columns = [name for name in [
    "Study Title", "Conditions", "Interventions", "Brief Summary",
    "Primary Outcome Measures"
] if name in df.columns]
df["combined_text"] = df[text_columns].fillna("").apply(
    lambda row: "\n\n".join(
        f"{name}: {str(row[name]).strip()}"
        for name in text_columns if str(row[name]).strip()
    ), axis=1
)
df = df.loc[df["combined_text"].str.strip().ne("")].reset_index(drop=True)
if df.empty:
    raise ValueError("The CSV has no searchable text.")

# Accept YYYY, YYYY-MM, and YYYY-MM-DD; never infer dates from summary text.
start_year = df["Start Date"].astype("string").str.extract(
    r"^\s*(\d{4})(?:-\d{2})?(?:-\d{2})?\s*$", expand=False
)
df["start_year"] = pd.to_numeric(start_year, errors="coerce").astype("Int64")
texts = df["combined_text"].tolist()
ids = [f"trial_{i}" for i in range(len(df))]
metadata = []
for _, row in df.iterrows():
    item = {
        "nct_id": "" if pd.isna(row["NCT Number"]) else str(row["NCT Number"]),
        "title": "" if pd.isna(row["Study Title"]) else str(row["Study Title"]),
        "start_date": "" if pd.isna(row["Start Date"]) else str(row["Start Date"]),
        "url": "" if pd.isna(row.get("Study URL")) else str(row.get("Study URL")),
    }
    if pd.notna(row["start_year"]):
        item["year"] = int(row["start_year"])
    metadata.append(item)

years = sorted(df["start_year"].dropna().astype(int).unique().tolist())
print(f"Loaded {len(texts)} trials from {CSV_PATH.name}")
print(f"Available start years: {years}")
print(f"Trials without start years: {df['start_year'].isna().sum()}")
display(df[["NCT Number", "Study Title", "Start Date", "start_year"]].head())


Loaded 837 trials from ctg-studies.csv
Available start years: [2007, 2008, 2009, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028]
Trials without start years: 1


,NCT Number,Study Title,Start Date,start_year
0,NCT07043361,"Efficacy of Semaglutide in Glycemic Control, W...",2023-12-18,2023
1,NCT06497049,Open Randomized Study of Comparative Pharmacok...,2023-06-30,2023
2,NCT06909006,Semaglutide Treatment in Type 1 Diabetes,2025-10,2025
3,NCT07732218,Efficacy of Once Weekly Semaglutide in Patient...,2022-07-04,2022
4,NCT06634927,"Pharmacokinetic Similarity, Safety, and Immuno...",2024-09-22,2024


In [2]:
from sentence_transformers import SentenceTransformer

# First run downloads pretrained weights and requires internet access.
models = {
    "general": SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"),
    "biomedical": SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO"),
}
print("Both embedding models loaded.")


/Users/jothigupta/Dropbox/AIPI 510/semaglutide-clinicial-trial-semantic-similarity/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 13:45:26.150639 22763 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27501.37it/s]


Both embedding models loaded.


In [3]:
import chromadb

# A fresh in-memory client avoids retaining previous notebook data.
client = chromadb.EphemeralClient()
collections = {}
for space_name, model in models.items():
    collection_name = f"ctg_search_{space_name}"
    # Reset only this notebook's collection when this cell is rerun.
    if collection_name in [item.name for item in client.list_collections()]:
        client.delete_collection(collection_name)
    collection = client.create_collection(
        name=collection_name, metadata={"hnsw:space": "cosine"}
    )
    collections[space_name] = collection
    # Batch inserts avoid Chroma's maximum batch size on larger CSVs.
    for start in range(0, len(texts), 128):
        stop = start + 128
        embeddings = model.encode(
            texts[start:stop], batch_size=16,
            normalize_embeddings=True, show_progress_bar=True
        )
        collection.add(
            ids=ids[start:stop], documents=texts[start:stop],
            embeddings=embeddings.tolist(), metadatas=metadata[start:stop]
        )
    print(f"{space_name}: indexed {collection.count()} trials")


Batches: 100%|██████████| 5/5 [00:00<00:00, 11.75it/s]


general: indexed 837 trials


Batches: 100%|██████████| 5/5 [00:02<00:00,  1.71it/s]

biomedical: indexed 837 trials


In [4]:
def multi_space_temporal_search(query, target_year=None):
    if not query or not str(query).strip():
        return "Enter a clinical trial search query."
    year = None
    if target_year not in (None, "", "All years", 0):
        try:
            number = float(target_year)
            if not number.is_integer():
                return "Choose a whole-number start year."
            year = int(number)
        except (ValueError, TypeError, OverflowError):
            return "Choose a valid start year."
    where_filter = {"year": year} if year is not None else None
    eligible = sum(1 for item in metadata if year is None or item.get("year") == year)
    if eligible == 0:
        return f"No trials have Start Date year {year}."

    output = []
    for space_name, model in models.items():
        kwargs = dict(
            query_embeddings=model.encode(
                [str(query).strip()], normalize_embeddings=True
            ).tolist(),
            n_results=1,
            include=["documents", "metadatas"]
        )
        if where_filter is not None:
            kwargs["where"] = where_filter
        results = collections[space_name].query(**kwargs)
        output.append(f"=== {space_name.upper()} ===")
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            output.append(
                f"{meta['nct_id']} | Start year: {meta.get('year', 'Unknown')}\n"
                f"{meta['title']}\n{meta['url']}\n{doc}\n"
            )
    return "\n".join(output)


In [5]:
import gradio as gr

demo = gr.Interface(
    fn=multi_space_temporal_search,
    inputs=[
        gr.Textbox(label="Clinical trial search query"),
        gr.Dropdown(
            choices=["All years"] + [str(year) for year in years],
            value="All years", label="Filter by trial start year"
        ),
    ],
    outputs=gr.Textbox(label="Most relevant study per model", lines=24),
    title="Clinical Trial Semantic Search",
    description=(
        "Return the single most relevant study from each embedding model. "
        "Year means the trial's Start Date year. All years includes trials with missing dates."
    ),
)
demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
